<a href="https://colab.research.google.com/github/Marianela-Fontana/dmeyf2026/blob/main/ENS_6300_6305.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

6.2 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> Python 3

Conectar la virtual machine donde esta corriendo Google Colab con el Google Drive, para poder tener persistencia de archivos


In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo


los siguientes comando estan en shell script de Linux

    Crear las carpetas en el Google Drive
    "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria kaggle de Python
    Bajar el dataset_pequeno al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab
    Bajar el dataset_historico al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"


In [1]:
require("data.table")

PARAM <- list()
PARAM$experimento <- "ENS_6300_6305"
PARAM$kaggle <- list()
PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(600, 1200, by = 50)   # ajustá a tu límite diario de envíos

# 1) leo las predicciones de los dos experimentos (revisá las rutas con ls)
archivos <- c(
  "buckets/b1/exp/WF6300/prediccion.txt",
  "buckets/b1/exp/WF6305/prediccion.txt"
)
tb_todas <- rbindlist(lapply(archivos, fread))

# 2) ensemble: promedio de probabilidad por cliente
tb_prediccion <- tb_todas[, list(prob = mean(prob)), by = numero_de_cliente]
setorder(tb_prediccion, -prob)

# 3) mismo bucle que el profe
dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle, sep = ","
  )

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'ensemble 6300+6302 envios=", envios, "'")

  linea <- paste(comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern = TRUE)
  cat(salida, "\n")
}

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


